# Прогноз задержки, риск и объяснения для frontend

Этот notebook обучает регрессор задержки и классификатор риска, затем объясняет прогнозы. Регрессор отвечает «сколько секунд задержки ожидается», классификатор — «какова вероятность задержки больше двух минут», SHAP — «какие признаки повлияли на прогноз регрессора».

## Как запускать

Из корня репозитория запустить `python ml_service/research/vovchik/train_from_notebook.py` или открыть notebook с рабочей папкой в корне репозитория и нажать **Run All**. Нужны `numpy`, `pandas`, `catboost`, `scipy`, `scikit-learn`, `nbformat`.

По умолчанию обе модели переобучаются. Артефакты исследования сохраняются в `ml_service/research/vovchik/results/`, конкурсный CSV — в `submission.csv` и `dataset/submission.csv`. После проверки нужно скопировать три файла моделей (`final_model.cbm`, `risk_model.cbm`, `risk_metadata.json`) в `ml_service/app/artifacts/`.

Чтобы повторить только обучение классификатора, установить `RETRAIN_REGRESSOR = False`; тогда загружается уже сохранённый регрессор из `results/`.

Блоки выполняются сверху вниз, потому что каждый использует результат предыдущего:
`CSV → таблицы признаков → фиксированный список признаков → регрессор → риск + SHAP + файлы`.

## 1. Загрузка данных

`DATA = Path("dataset")` — путь внутри этого репозитория. Запускать из его корня.

Ожидаемые файлы:

```text
dataset/
├── train/traffic.csv, schedule.csv
├── test/traffic.csv, schedule.csv
├── labels/labels_train.csv, labels_test.csv
├── validate/traffic.csv, schedule_plan.csv, points.csv
└── sample_submission.csv
```

- `train` содержит признаки и реальные задержки для обучения.
- `test` содержит ответы для локальной оценки качества.
- `validate` содержит только входные данные: реальные ответы скрыты, здесь формируем итоговые прогнозы.

`T` — момент формирования прогноза. `target_time_begin` — плановое прибытие на целевую остановку через 10–15 минут; `cur_dev_s` — известная текущая задержка; `target_delay_s` — ответ, доступный только в train/test. Положительная задержка означает опоздание, отрицательная — опережение графика.

Функция `read_split` читает точки прогноза, плановое расписание и телеметрию. Фактические времена из расписания не загружаются в признаки. Вывод `Samples` показывает число точек в каждом наборе.

In [27]:
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

DATA = Path("dataset")
assert DATA.is_dir()
PLAN_COLS = ["tt_action_item_id", "tr_id", "time_begin", "geom"]
TRAFFIC_COLS = ["packet_id", "tr_id", "unit_id", "event_time", "gps_time",
                "speed", "receive_time", "is_hist_data", "location_valid", "lon", "lat"]

def read_split(split):
    points_path = DATA / ("validate/points.csv" if split == "validate"
                          else f"labels/labels_{split}.csv")
    points = pd.read_csv(points_path, dtype={"sample_id": str, "tr_id": str,
                                              "target_stop_id": str})
    plan_path = DATA / ("validate/schedule_plan.csv" if split == "validate"
                        else f"{split}/schedule.csv")
    # Фактические времена не читаем.
    plan = pd.read_csv(plan_path, usecols=PLAN_COLS,
                       dtype={"tr_id": str, "tt_action_item_id": str})
    traffic = pd.read_csv(DATA / split / "traffic.csv", usecols=TRAFFIC_COLS,
                          dtype={"packet_id": str, "tr_id": str, "unit_id": str},
                          low_memory=False)
    traffic = traffic.loc[traffic["tr_id"].isin(points["tr_id"])].copy()
    return points.reset_index(drop=True), plan, traffic

train, plan_train, traffic_train = read_split("train")
test, plan_test, traffic_test = read_split("test")
validate, plan_validate, traffic_validate = read_split("validate")
print("Samples:", len(train), len(test), len(validate))

Samples: 4434 353 151


## 2. Расчёт признаков на момент T

Для каждой точки собираем признаки движения: скорость и её изменение, долю и продолжительность простоя, возраст телеметрии, расстояние по прямой до целевой остановки, скорость приближения, текущую задержку и плановое расписание.

- `telemetry_at` рассчитывает статистику движения за окна от 30 секунд до 15 минут.
- `spatial_at` рассчитывает признаки положения и приближения к цели.
- `build_features` объединяет их с данными точки прогноза и расписания.

**Защита от будущего:** пакет используется только если одновременно `event_time <= T` и `receive_time <= T`: событие уже произошло и пакет уже получен. Плановое будущее расписание известно заранее, поэтому его использовать можно. Реальная будущая задержка и фактическое будущее прибытие в признаки не входят.

Результат — `F_train`, `F_test`, `F_validate`. В этих таблицах есть признаки и служебные поля; целевые ответы хранятся отдельно. Пропуски числовых признаков могут быть нормальны, например при отсутствии подходящей GPS-истории. Проверки `assert` контролируют соответствие остановок, порядок и число точек, отсутствие ответов среди признаков.

In [28]:
STOP_SPEED_KMH = 2.0
WINDOWS = {"30s": 30, "1m": 60, "3m": 180, "5m": 300, "10m": 600, "15m": 900}

def haversine_m(lon1, lat1, lon2, lat2):
    r = 6_371_008.8
    a1, b1, a2, b2 = map(np.radians, [lon1, lat1, lon2, lat2])
    a = np.sin((b2-b1)/2)**2 + np.cos(b1)*np.cos(b2)*np.sin((a2-a1)/2)**2
    return 2*r*np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def telemetry_at(history, now):
    out = {"unit_id": "missing", "packet_count_15m": 0, "packet_frequency": 0.0}
    if history.empty:
        return out
    last = history.iloc[-1]
    recent = history.loc[history["event_time"] > now - pd.Timedelta(minutes=15)]
    known_speed = history.dropna(subset=["speed"])
    out["unit_id"] = str(last["unit_id"]) if pd.notna(last["unit_id"]) else "missing"
    for col in ("event_time", "receive_time", "gps_time"):
        value = last[col]
        out[col] = value.value / 1e9 if pd.notna(value) else np.nan
    out["speed_now"] = known_speed["speed"].iloc[-1] if len(known_speed) else np.nan
    out["telemetry_age"] = (now - last["event_time"]).total_seconds()
    known_gps = history["gps_time"].dropna()
    out["gps_age"] = (now - known_gps.iloc[-1]).total_seconds() if len(known_gps) else np.nan
    out["receive_delay"] = (last["receive_time"] - last["event_time"]).total_seconds()
    out["last_packet_is_hist"] = float(last["is_hist_data"]) if pd.notna(last["is_hist_data"]) else np.nan
    out["packet_count_15m"] = len(recent)
    out["packet_frequency"] = len(recent) / 15
    out["hist_packet_fraction"] = recent["is_hist_data"].mean() if len(recent) else np.nan
    out["mean_receive_delay"] = ((recent["receive_time"] - recent["event_time"])
                                 .dt.total_seconds().mean() if len(recent) else np.nan)
    out["max_packet_gap"] = recent["event_time"].diff().dt.total_seconds().max()
    for suffix, seconds in WINDOWS.items():
        window = recent.loc[recent["event_time"] > now - pd.Timedelta(seconds=seconds)]
        valid = window.dropna(subset=["speed"])
        speeds = valid["speed"]
        for stat in ("mean", "std", "min", "max", "median"):
            out[f"speed_{stat}_{suffix}"] = getattr(speeds, stat)() if len(speeds) else np.nan
        if suffix in ("1m", "3m", "5m"):
            out[f"stopped_fraction_{suffix}"] = (speeds <= STOP_SPEED_KMH).mean() if len(speeds) else np.nan
            if len(valid) >= 2:
                span = (valid["event_time"].iloc[-1] - valid["event_time"].iloc[0]).total_seconds() / 60
                out[f"speed_slope_{suffix}"] = ((speeds.iloc[-1] - speeds.iloc[0]) / span
                                                 if span > 0 else np.nan)
        if suffix == "5m":
            longest, start = 0.0, None
            for event_time, speed in zip(valid["event_time"], speeds):
                if speed <= STOP_SPEED_KMH:
                    if start is None:
                        start = event_time
                    longest = max(longest, (event_time - start).total_seconds())
                else:
                    start = None
            out["longest_stop_5m"] = longest if len(valid) else np.nan
        if suffix == "10m":
            stopped = speeds <= STOP_SPEED_KMH
            out["stop_count_10m"] = int((stopped & ~stopped.shift(fill_value=False)).sum()) if len(valid) else 0
    if len(known_speed):
        moving = known_speed.loc[known_speed["speed"] > STOP_SPEED_KMH]
        out["time_since_last_move"] = ((now - moving["event_time"].iloc[-1]).total_seconds()
                                       if len(moving) else np.nan)
        if known_speed["speed"].iloc[-1] <= STOP_SPEED_KMH:
            prior_move = np.flatnonzero(known_speed["speed"].to_numpy() > STOP_SPEED_KMH)
            start_idx = prior_move[-1] + 1 if len(prior_move) else 0
            out["current_stop_duration"] = (now - known_speed["event_time"].iloc[start_idx]).total_seconds()
        else:
            out["current_stop_duration"] = 0.0
    return out

def spatial_at(history, now, target_lon, target_lat):
    out = {"sp_has_position": 0}
    valid = history.loc[history["location_valid"].eq(True)
                        & history["lon"].between(-180, 180)
                        & history["lat"].between(-90, 90)
                        & ~(history["lon"].eq(0) & history["lat"].eq(0))]
    if valid.empty:
        return out
    valid = valid.drop_duplicates("event_time", keep="last")
    current = valid.iloc[-1]
    age = (now - current["event_time"]).total_seconds()
    distance = haversine_m(current["lon"], current["lat"], target_lon, target_lat)
    out.update(sp_has_position=1, sp_position_age_s=age, sp_distance_to_target_m=distance)
    for minutes in (1, 3):
        query = now - pd.Timedelta(minutes=minutes)
        previous = valid.loc[valid["event_time"] <= query]
        if previous.empty or age > 60:
            continue
        past = previous.iloc[-1]
        if (query - past["event_time"]).total_seconds() > 60:
            continue
        elapsed = (current["event_time"] - past["event_time"]).total_seconds()
        if elapsed <= 0:
            continue
        old_distance = haversine_m(past["lon"], past["lat"], target_lon, target_lat)
        approach = old_distance - distance
        out[f"sp_approach_{minutes}m_m"] = approach
        out[f"sp_approach_speed_{minutes}m_mps"] = approach / elapsed
    return out

def build_features(points, plan, traffic):
    samples = points[["sample_id", "tr_id", "T", "target_stop_id",
                      "target_time_begin", "cur_dev_s"]].copy()
    for col in ("T", "target_time_begin"):
        samples[col] = pd.to_datetime(samples[col], format="mixed")
    plan = plan.copy()
    plan["time_begin"] = pd.to_datetime(plan["time_begin"], format="mixed")
    coords = plan["geom"].str.extract(r"POINT\s*\(\s*([-\d.]+)\s+([-\d.]+)\s*\)")
    plan["target_lon"] = pd.to_numeric(coords[0])
    plan["target_lat"] = pd.to_numeric(coords[1])
    target = plan[["tr_id", "tt_action_item_id", "time_begin", "target_lon", "target_lat"]].rename(
        columns={"tt_action_item_id": "target_stop_id", "time_begin": "target_plan_time"})
    samples = samples.merge(target, on=["tr_id", "target_stop_id"], how="left",
                            validate="many_to_one", sort=False)
    assert samples["target_lon"].notna().all() and samples["target_lat"].notna().all()
    assert samples["target_time_begin"].eq(samples["target_plan_time"]).all()
    samples["hour"] = samples["T"].dt.hour
    samples["minute"] = samples["T"].dt.minute
    samples["minute_of_day"] = samples["hour"]*60 + samples["minute"]
    samples["day_of_week"] = samples["T"].dt.dayofweek
    samples["is_weekend"] = (samples["day_of_week"] >= 5).astype(int)
    samples["sin_time"] = np.sin(2*np.pi*samples["minute_of_day"]/1440)
    samples["cos_time"] = np.cos(2*np.pi*samples["minute_of_day"]/1440)
    samples["target_hour"] = samples["target_time_begin"].dt.hour
    samples["target_minute_of_day"] = (samples["target_hour"]*60
                                      + samples["target_time_begin"].dt.minute)
    samples["forecast_horizon_s"] = (samples["target_time_begin"] - samples["T"]).dt.total_seconds()
    samples["abs_cur_dev"] = samples["cur_dev_s"].abs()
    samples["is_late_now"] = (samples["cur_dev_s"] > 0).astype(int)
    samples["is_early_now"] = (samples["cur_dev_s"] < 0).astype(int)
    by_plan = {trip: group.sort_values("time_begin") for trip, group in plan.groupby("tr_id")}
    samples["stops_to_target"] = [
        by_plan[row["tr_id"]].loc[
            (by_plan[row["tr_id"]]["time_begin"] > row["T"])
            & (by_plan[row["tr_id"]]["time_begin"] <= row["target_plan_time"]),
            "tt_action_item_id"].nunique()
        for _, row in samples.iterrows()
    ]

    traffic = traffic.copy()
    for col in ("event_time", "receive_time", "gps_time"):
        traffic[col] = pd.to_datetime(traffic[col], format="mixed", errors="coerce")
    traffic = traffic.dropna(subset=["event_time", "receive_time"])
    traffic = traffic.drop_duplicates(["tr_id", "packet_id"])
    by_trip = {trip: group.sort_values(["event_time", "packet_id"]).reset_index(drop=True)
               for trip, group in traffic.groupby("tr_id")}
    rows = []
    for _, row in samples.iterrows():
        now = row["T"]
        trip = by_trip.get(row["tr_id"])
        available = (trip.loc[(trip["event_time"] <= now) & (trip["receive_time"] <= now)]
                     if trip is not None else traffic.iloc[0:0])
        assert (available["event_time"] <= now).all() and (available["receive_time"] <= now).all()
        one = telemetry_at(available, now)
        one.update(spatial_at(available, now, row["target_lon"], row["target_lat"]))
        rows.append(one)
    result = pd.concat([samples.reset_index(drop=True),
                        pd.DataFrame.from_records(rows)], axis=1)
    assert len(result) == len(points) and result["sample_id"].is_unique
    assert not {"target_delay_s", "target_class", "time_fact_begin"}.intersection(result.columns)
    return result

F_train = build_features(train, plan_train, traffic_train)
F_test = build_features(test, plan_test, traffic_test)
F_validate = build_features(validate, plan_validate, traffic_validate)
assert F_train["sample_id"].equals(train["sample_id"])
assert F_test["sample_id"].equals(test["sample_id"])
assert F_validate["sample_id"].equals(validate["sample_id"])
print("Feature frames:", F_train.shape, F_test.shape, F_validate.shape)


Feature frames: (4434, 84) (353, 84) (151, 84)


## 3. Фиксированный набор входов модели

Модель использует **78 признаков**: 17 из расписания/текущего состояния, 7 пространственных и 54 телеметрических. `FEATURE_COLS` задаёт их точный порядок.

Этот набор используется и основной моделью, и отдельным классификатором риска. Нельзя произвольно удалять или переименовывать признаки при загрузке готовой модели. Служебный `sample_id` нужен для сопоставления результатов, но не подаётся в модель.

In [29]:
SCHEDULE = ["tr_id", "target_stop_id", "hour", "minute", "minute_of_day",
            "day_of_week", "is_weekend", "sin_time", "cos_time", "target_hour",
            "target_minute_of_day", "forecast_horizon_s", "cur_dev_s",
            "abs_cur_dev", "is_late_now", "is_early_now", "stops_to_target"]
SPATIAL = ["sp_has_position", "sp_position_age_s", "sp_distance_to_target_m",
           "sp_approach_1m_m", "sp_approach_speed_1m_mps",
           "sp_approach_3m_m", "sp_approach_speed_3m_mps"]
TELEMETRY_54 = [
    "unit_id", "event_time", "receive_time", "gps_time", "speed_now",
    "telemetry_age", "gps_age", "receive_delay", "last_packet_is_hist",
    "packet_count_15m", "packet_frequency", "hist_packet_fraction",
    "mean_receive_delay", "max_packet_gap",
] + [f"speed_{stat}_{window}" for window in WINDOWS
     for stat in ("mean", "std", "min", "max", "median")] + [
    "stopped_fraction_1m", "stopped_fraction_3m", "stopped_fraction_5m",
    "speed_slope_1m", "speed_slope_3m", "speed_slope_5m",
    "longest_stop_5m", "stop_count_10m",
    "time_since_last_move", "current_stop_duration",
]
FEATURE_COLS = SCHEDULE + SPATIAL + TELEMETRY_54

## 4. Основная модель: загрузка, прогноз и MAE

При **`RETRAIN_REGRESSOR = True`** обучаем и сохраняем регрессор. Функция `model_input` выбирает признаки в заданном порядке и приводит категориальные идентификаторы к строкам.

Особенность регрессора: он предсказывает **поправку к текущей задержке**, поэтому итог считается так:

```text
predicted_delay_s = cur_dev_s + model.predict(features)
```

На test считаем абсолютную ошибку `abs(фактическая задержка − прогноз)`, а её среднее — **MAE в секундах**: меньше — лучше. Это локальная метрика, не скор скрытой таблицы хакатона. Для validate MAE посчитать нельзя, поскольку его ответы не предоставлены.

Далее формируем `submission` в памяти, сохраняя порядок ID из шаблона, и сохраняем его в корне при переобучении. Вероятность и SHAP в конкурсный CSV не добавляются.

In [ ]:
CATEGORICAL = ["tr_id", "target_stop_id", "unit_id"]
def model_input(frame):
    x = frame[FEATURE_COLS].copy()
    for col in CATEGORICAL:
        x[col] = x[col].fillna("missing").astype(str)
    return x


# По умолчанию сохраняем основную модель. True явно включает переобучение.
RETRAIN_REGRESSOR = True
MODEL_PATH = Path("ml_service/research/vovchik/results/final_model.cbm")

if RETRAIN_REGRESSOR:
    model = CatBoostRegressor(loss_function="MAE",
                              iterations=200,
                              depth=7,
                              learning_rate=0.05,
                              l2_leaf_reg=5,
                              random_seed=42,
                              verbose=False,
                              allow_writing_files=False)
    model.fit(model_input(F_train),
              train["target_delay_s"].to_numpy() - train["cur_dev_s"].to_numpy(),
              cat_features=CATEGORICAL)
    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    model.save_model(str(MODEL_PATH))
    print("Основная модель переобучена и сохранена")
else:
    if not MODEL_PATH.is_file():
        raise FileNotFoundError(
            f"Готовая модель не найдена: {MODEL_PATH.resolve()}. "
            "Сначала обучите регрессор с RETRAIN_REGRESSOR = True. "
            "Переобучение автоматически не запускается."
        )
    model = CatBoostRegressor()
    model.load_model(str(MODEL_PATH))
    print("Готовая основная модель загружена без переобучения")

test_prediction = test["cur_dev_s"].to_numpy() + model.predict(model_input(F_test))
test_mae = np.mean(np.abs(test["target_delay_s"].to_numpy() - test_prediction))
print(f"Local test MAE: {test_mae:.4f} seconds")
prediction = validate["cur_dev_s"].to_numpy() + model.predict(model_input(F_validate))
pred = pd.DataFrame({"sample_id": F_validate["sample_id"], "prediction": prediction})
submission = pd.read_csv(DATA / "sample_submission.csv", sep=";", dtype={"sample_id": str})[["sample_id"]]
submission = submission.merge(pred, on="sample_id", how="left", validate="one_to_one")
assert len(submission) == len(validate) and np.isfinite(submission["prediction"]).all()
if RETRAIN_REGRESSOR:
    submission.to_csv("submission.csv", sep=";", index=False)
# Последняя ячейка сохраняет полную копию submission в results и dataset.

## 5. Отдельная модель риска, SHAP и экспорт

`run_pipeline` из `ml_risk.py` выполняет три задачи.

### Вероятность задержки больше 120 секунд

Обучается отдельный CatBoostClassifier с целью `target_delay_s > 120` на тех же 78 признаках. Его прогноз не изменяет результат основного регрессора. Например, `delayProbability = 0.72` означает оценку вероятности **задержки больше двух минут**, а не «точность модели 72%» и не вероятность любой задержки.

Train делится по уникальным временам T на 60% / 20% / 20%: обучение классификатора, обучение sigmoid-калибровки, выбор между исходной и скорректированной вероятностью по меньшему Brier. Строки, ответ которых ещё неизвестен к следующей временной границе, исключаются из предыдущего участка. Поэтому доли строк после исключения могут отличаться от 60/20/20.

Калибровка проверяет соответствие процентов наблюдаемой частоте событий. Test не используется для обучения и программного выбора калибровки. В отчёте сохраняются оба варианта вероятностей, выбранный вариант, ROC-AUC/AP (различение событий), Brier/log loss (ошибка вероятностей), ECE и интервалы калибровки. Меньший Brier сам по себе не доказывает идеальную калибровку.

### Объяснения конкретного прогноза

SHAP вычисляет вклад **каждого признака регрессора** в секундах. Для интерфейса выбираются до трёх самых больших по модулю ненулевых вкладов: машинное имя `feature`, русская подпись `label`, значение `value`, вклад `shapValueSec`.

Положительный вклад увеличивает прогноз задержки, отрицательный уменьшает. Это объяснение регрессора, не вероятности классификатора и не доказанная причина сбоя. Простой сам по себе не доказывает длительную посадку или пробку.

Проверяем равенство без округления:

```text
predictionSec = currentDeviationSec + baseValueSec
              + сумма показанных shapValueSec + remainingShapValueSec
```

`currentDeviationSec` — текущая задержка, добавленная вне модели; `baseValueSec` — базовое значение SHAP для поправки; `remainingShapValueSec` — сумма вкладов остальных признаков. Без этих частей три показанных фактора обычно не восстановят весь прогноз.

### Сохранение результатов

Экспортируются все точки validate. В CSV проверяются две колонки `sample_id;prediction`, отсутствие пропусков/дублей, конечность прогнозов и точное совпадение ID с validate и шаблоном. Все проверки должны пройти до передачи файла команде.

In [ ]:
from ml_service.app.services.ml_risk import run_pipeline

risk_report = run_pipeline(
    model, F_train, train, F_test, test, F_validate, validate, DATA,
    output_dir=MODEL_PATH.parent, regressor_path=MODEL_PATH,
)


## Что передать команде после запуска

| Файл | Для чего нужен |
|---|---|
| `results/final_model.cbm` | Обученный регрессор |
| `results/risk_model.cbm` | Отдельная модель риска |
| `results/risk_metadata.json` | Порядок признаков, калибровка, выбранный метод и хеш регрессора |
| `results/predictions_validate.json` | Прогноз, вероятность и SHAP для каждой точки; frontend объединяет записи по `sampleId` |
| `results/metrics.json` | Метрики риска и регрессии, таблицы калибровки |
| `results/test_predictions.csv` | Прогноз, реальный ответ и абсолютная ошибка для каждой test-точки |
| `results/submission.csv`, `submission.csv`, `DATA / "submission.csv"` | Полный конкурсный файл; в текущем датасете 151 строка данных |
| `ML_RISK.md` | Контракт полей и пример загрузки моделей для backend |

В JSON `predictionSec` и `predicted_delay_s` — один прогноз в секундах; `delayProbability` — число от 0 до 1 (умножить на 100 для процентов); `explanation.factors` — объяснения. Значения округлять только при отображении. Notebook готовит данные, но сам не меняет интерфейс и не запускает сервер.

### Как интерпретировать результаты

На проверенном наборе MAE основного регрессора около **63,88 с**, ROC-AUC риска около **0,915**. Это ориентиры последнего проверенного запуска, а не обещание для любой версии данных или модели; актуальные числа выводятся выше и сохраняются в `metrics.json`.

На train выбрана sigmoid-калибровка, однако на test она немного хуже исходных вероятностей по Brier. В отчёте это не скрывается. Test просматривался в ходе разработки, содержит те же даты и пересекающиеся ТС с train, поэтому эти метрики не доказывают качество на следующем дне или новых ТС. Для такой оценки нужен независимый временной набор.